In [1]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
import time

2026-02-24 10:43:28.833740494 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


### Embedding Model

In [2]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
model.get_sentence_embedding_dimension()

384

In [4]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

### Simple Embedding

In [5]:
document = """This study examines the environmental benefits of 
electric bus fleets in three major metropolitan areas over a 
two-year period. Our analysis shows that electric buses reduce 
carbon emissions by an average of 65% compared to traditional 
diesel buses, while also decreasing noise pollution in urban 
centers by 40 decibels."""

In [6]:
document_embeddings = model.encode([document])

In [7]:
document_embeddings.shape

(1, 384)

### Qdrant

In [9]:
collection_name = "qdrant-tests2"
vector_name = "all-MiniLM-L6-v2"

In [10]:
client = QdrantClient("http://localhost:6333")

In [11]:
client.delete_collection(collection_name)

True

In [12]:
client.create_collection(
    collection_name,
    vectors_config={
        vector_name: models.VectorParams(
            size=model.get_sentence_embedding_dimension(),
            distance=models.Distance.COSINE,
        ),       
    }
)

True

In [13]:
documents = [
    "Qdrant is a vector database designed for similarity search applications",
    "SQL databases use structured tables with predefined schemas for data storage",
    "Using Qdrant you can store embeddings and perform efficient searches",
    "Traditional SQL queries filter data using exact matches and joins",
    "Qdrant supports multi-vector configurations for late interaction models like ColBERT",
    "SQL performs well for transactional workloads but lacks semantic search capabilities",
    "The Qdrant client allows you to create collections with custom distance metrics",
    "Migrating from SQL to vector databases enables similarity-based retrieval at scale",
    "Qdrant's MaxSim comparator enables token-level similarity scoring for multi-vectors",
    "SQL databases struggle with high-dimensional embeddings unlike specialized vector stores",
]

In [14]:
client.upsert(
    collection_name,
    points=[
        models.PointStruct(
            id=i,
            vector={
                vector_name: model.encode([document])[0],
            },
            payload={"text": document},
        )
        for i, document in enumerate(documents, start=1)
    ],
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

### Query

In [15]:
def dense_query(q: str, limit: int = 5) -> list[dict]:
    start_time = time.monotonic()
    embedding = model.encode([q])[0]
    end_time = time.monotonic()
    
    print("Dense vector generation time:", end_time - start_time)

    start_time = time.monotonic()
    result = client.query_points(
        collection_name,
        query=embedding,
        using=vector_name,
        limit=limit,
        with_payload=True,
    )
    end_time = time.monotonic()
    print("Query time:", end_time - start_time)

    return [point.payload for point in result.points]    

In [16]:
query = "search performance in Qdrant"

In [17]:
dense_hits = dense_query(query)

Dense vector generation time: 0.012345557000116969
Query time: 0.011995666000075289


In [18]:
dense_hits

[{'text': 'Using Qdrant you can store embeddings and perform efficient searches'},
 {'text': 'Qdrant is a vector database designed for similarity search applications'},
 {'text': 'Qdrant supports multi-vector configurations for late interaction models like ColBERT'},
 {'text': "Qdrant's MaxSim comparator enables token-level similarity scoring for multi-vectors"},
 {'text': 'The Qdrant client allows you to create collections with custom distance metrics'}]